# 画素の位置で LC4 と LPLC2 を刺激する

`poster/poster.md` の式を、コネクトームの列で数値にする。青は LC4、赤は LPLC2 である。どちらも、色のついた画素は 1 のまま、周りへ広げる。

画素 $x$ から一番近いその色の画素までの距離を $d(x)$ とする。距離は画素数を画像の横幅で割った長さで、縦も横幅を単位にする。広がりは $\sigma_d = 0.15$ とする。これは横 10 px の並びで、隣の画素が約 0.8 になる値である。ポスターの式では $\sigma_d$ は記号のままである。

$$
a(x)=\exp\left(-\frac12\left(\frac{d(x)}{\sigma_d}\right)^2\right)
$$

$f_i$ は、その細胞の列での $a(x)$ の平均である。列に後シナプスが $n$ 本あるときは、$n$ で重みを付ける。$w_i$ は、その細胞から同側の DNp01 への化学シナプス数である。$F$ は $\sum w_i f_i$ を、その細胞種のその側の $w_i$ の和で割ったものである。色の画素だけを 1 にする場合も、同じ平均で並べる。

In [1]:
from pathlib import Path
import sys

import cv2
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "flycar").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from flycar.loom import cells

SIGMA_D = 0.15
WIDTH = 320
HEIGHT = 180


def spread(mask: np.ndarray, sigma_d: float = SIGMA_D) -> np.ndarray:
    """Nearest-color weight. Colored pixels stay 1. Distance is in image widths."""
    color = np.asarray(mask, dtype=bool)
    width = color.shape[1]
    # distanceTransform measures the distance to the nearest zero, so color is zero.
    src = np.where(color, 0, 255).astype(np.uint8)
    dist = cv2.distanceTransform(src, cv2.DIST_L2, 5) / width
    return np.exp(-0.5 * (dist / sigma_d) ** 2)


def fraction(pop, field: np.ndarray) -> np.ndarray:
    """Return F on the left and the right. The denominator is that type's own synapses."""
    height, width = field.shape
    ui = np.clip(np.rint(pop.u * width - 0.5).astype(np.int64), 0, width - 1)
    vi = np.clip(np.rint(pop.v * height - 0.5).astype(np.int64), 0, height - 1)
    summed = pop._weighted(field[vi, ui])
    return summed / pop.side_sum


## 横 10 px、縦 1 px

並びは白が五つ、色が二つ、白が三つである。青でも赤でも、画素上の $a(x)$ は同じ形になる。

In [2]:
toy = np.zeros((1, 10), dtype=bool)
toy[0, 5:7] = True
labels = ["白", "白", "白", "白", "白", "色", "色", "白", "白", "白"]
values = spread(toy)[0]
for label, value in zip(labels, values):
    print(f"{label} {value:.3f}")


白 0.004
白 0.029
白 0.135
白 0.411
白 0.801
色 1.000
色 1.000
白 0.801
白 0.411
白 0.135


## 列での $F$

上の並びを、横 320 px、縦 180 px の画面いっぱいに伸ばす。色は $u$ が 0.5 から 0.7 の帯で、右半分の左端から始まる。左半分にその色の画素は無い。

LC4 には青の帯を、LPLC2 には同じ位置の赤の帯を入れる。広がりのない列は、画素が色のときだけ 1 である。LPLC2 の画面位置は、小葉板の接合部が乗っているマス目の平均で、車が読む `loom_cells.npz` にある。

In [3]:
mask = np.zeros((HEIGHT, WIDTH), dtype=bool)
x0 = int(round(0.5 * WIDTH))
x1 = int(round(0.7 * WIDTH))
mask[:, x0:x1] = True
soft = spread(mask)
hard = mask.astype(np.float64)
pack = cells()

print(f"帯 u=[0.5, 0.7)  σ_d={SIGMA_D}  画像 {HEIGHT}x{WIDTH}")
for name, pop in (("LC4 青", pack.lc4), ("LPLC2 赤", pack.lplc2)):
    painted = fraction(pop, hard)
    spread_f = fraction(pop, soft)
    print(
        f"{name}  画素だけ L {painted[0]:.4f} R {painted[1]:.4f}"
        f"    広がり L {spread_f[0]:.4f} R {spread_f[1]:.4f}"
    )

row = soft[HEIGHT // 2]
print("画面中央の行の a(u)")
for u in (0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8):
    x = int(np.clip(np.rint(u * WIDTH - 0.5), 0, WIDTH - 1))
    print(f"  u={u:.1f}  a={row[x]:.3f}  色={bool(mask[HEIGHT // 2, x])}")


帯 u=[0.5, 0.7)  σ_d=0.15  画像 180x320
LC4 青  画素だけ L 0.0007 R 0.3341    広がり L 0.3211 R 0.8024
LPLC2 赤  画素だけ L 0.0001 R 0.1487    広がり L 0.2256 R 0.7156
画面中央の行の a(u)
  u=0.0  a=0.004  色=False
  u=0.2  a=0.135  色=False
  u=0.3  a=0.411  色=False
  u=0.4  a=0.801  色=False
  u=0.5  a=1.000  色=True
  u=0.6  a=1.000  色=True
  u=0.8  a=0.790  色=False
